In [2]:
import os
# print(os.getcwd())
os.chdir(r'C:\Users\awet0') # Change to your own enviroment


import pandas as pd
import numpy as np
import math
from IPython.display import display
import ace_tools_open as tools
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display






# Read the CSV file (using ";" as the delimiter)
df  = pd.read_csv("OneDrive/ACIT/Master Thesis/JupytherLab/cloudsim-custom-implementation/logs/exp1/PA_Storage.csv", delimiter=";", low_memory=False)



# === STEP 1: Convert data types ===
df['cpu_power'] = df['cpu_power'].astype(float)
df['ram_power'] = df['ram_power'].astype(float)
df['bw_power'] = df['bw_power'].astype(float)
df['I/O_power'] = df['I/O_power'].astype(float)

df['mips'] = df['mips'].astype(float)
df['available_mips'] = df['available_mips'].astype(float)
df['cpu_utilization'] = df['cpu_utilization'].astype(float)

df['ram'] = df['ram'].astype(float)
df['available_ram'] = df['available_ram'].astype(float)
df['bw'] = df['bw'].astype(float)
df['available_bw'] = df['available_bw'].astype(float)
df['storage'] = df['storage'].astype(float)
df['available_storage'] = df['available_storage'].astype(float)
df['disk_I/O'] = df['disk_I/O'].astype(float)

# === STEP 2: Loop through each datacenter ===
summaries = []
for dc_name, group in df.groupby('datacenter_name'):
    active_hosts_df = group[group['active'] == True]
    power_on_hosts_df = group[group['power_on'] == True]

    def usage_percentage(available, total):
        return [(t - a) / t * 100 if t else 0 for a, t in zip(available, total)]

    cpu_usage_pct = usage_percentage(group['available_mips'], group['mips'])
    ram_usage_pct = usage_percentage(group['available_ram'], group['ram'])
    bw_usage_pct = usage_percentage(group['available_bw'], group['bw'])
    storage_usage_pct = usage_percentage(group['available_storage'], group['storage'])

    total_cpu_power = group['cpu_power'].sum()
    total_ram_power = group['ram_power'].sum()
    total_bw_power = group['bw_power'].sum()
    total_io_power = group['I/O_power'].sum()
    total_disk_io = group['disk_I/O'].sum()

    def parse_vms(vms_str):
        vms_data = []
        if pd.isna(vms_str) or vms_str.strip() == '':
            return vms_data
        for vm_entry in vms_str.strip(':').split(':'):
            try:
                dc_id, mips, ram, bw = vm_entry.split(',')
                vms_data.append({
                    'datacenter_id': int(dc_id),
                    'allocated_mips': float(mips),
                    'requested_ram': float(ram),
                    'requested_bw': float(bw)
                })
            except:
                continue
        return vms_data

    all_vms = []
    for index, row in group.iterrows():
        vms = parse_vms(row['vms'])
        for vm in vms:
            vm['time'] = row['time']
            vm['host_id'] = row['host_id']
            all_vms.append(vm)

    vms_df = pd.DataFrame(all_vms)

    total_mips = group['mips'].sum()
    available_mips = group['available_mips'].sum()
    mips_usage_pct_dc = (total_mips - available_mips) / total_mips * 100 if total_mips else 0

    total_ram = group['ram'].sum()
    available_ram = group['available_ram'].sum()
    ram_usage_pct_dc = (total_ram - available_ram) / total_ram * 100 if total_ram else 0

    total_bw = group['bw'].sum()
    available_bw = group['available_bw'].sum()
    bw_usage_pct_dc = (total_bw - available_bw) / total_bw * 100 if total_bw else 0

    total_storage = group['storage'].sum()
    available_storage = group['available_storage'].sum()
    storage_usage_pct_dc = (total_storage - available_storage) / total_storage * 100 if total_storage else 0

    group['time_hours'] = group['time'] / 3600
    total_cpu_energy_kwh = 0.0
    total_ram_energy_kwh = 0.0
    total_bw_energy_kwh = 0.0
    total_io_energy_kwh = 0.0

    for host_id, subgroup in group.groupby('host_id'):
        subgroup_sorted = subgroup.sort_values(by='time_hours')
        times = subgroup_sorted['time_hours'].values
        cpu_powers = subgroup_sorted['cpu_power'].values
        ram_powers = subgroup_sorted['ram_power'].values
        bw_powers = subgroup_sorted['bw_power'].values
        io_powers = subgroup_sorted['I/O_power'].values

        for i in range(1, len(times)):
            delta_t = times[i] - times[i - 1]
            total_cpu_energy_kwh += 0.5 * (cpu_powers[i] + cpu_powers[i - 1]) * delta_t / 1000
            total_ram_energy_kwh += 0.5 * (ram_powers[i] + ram_powers[i - 1]) * delta_t / 1000
            total_bw_energy_kwh += 0.5 * (bw_powers[i] + bw_powers[i - 1]) * delta_t / 1000
            total_io_energy_kwh += 0.5 * (io_powers[i] + io_powers[i - 1]) * delta_t / 1000

    used_mips = total_mips - available_mips
    cpu_utilization_pct = (used_mips / total_mips * 100) if total_mips else 0
    avg_cpu_util_column = group['cpu_utilization'].mean() * 100
    power_consumption_total = total_cpu_energy_kwh + total_ram_energy_kwh + total_bw_energy_kwh + total_io_energy_kwh

    summary = {
        "Datacenter": dc_name,
        "Total Active Hosts (Running)": active_hosts_df['host_id'].nunique(),
        "Total Hosts Powered On": power_on_hosts_df['host_id'].nunique(),
        "Total VMs Parsed": len(vms_df),
        "Total MIPS": total_mips,
        "Used MIPS": used_mips,
        "Total CPU Power (W)": total_cpu_power,
        "Total RAM Power (W)": total_ram_power,
        "Total BW Power (W)": total_bw_power,
        "Total I/O Power (W)": total_io_power,
        "Total Disk I/O Throughput MB": total_disk_io,
        "Total CPU Energy (kWh)": total_cpu_energy_kwh,
        "Total RAM Energy (kWh)": total_ram_energy_kwh,
        "Total BW Energy (kWh)": total_bw_energy_kwh,
        "Total I/O Energy (kWh)": total_io_energy_kwh,
        "Total Power Consumption (kWh)": power_consumption_total,
        "Datacenter MIPS Usage (%)": mips_usage_pct_dc,
        "Datacenter RAM Usage (%)": ram_usage_pct_dc,
        "Datacenter Network BW Usage (%)": bw_usage_pct_dc,
        "Datacenter Storage Usage (%)": storage_usage_pct_dc
    }

    summaries.append(summary)

summary_df = pd.DataFrame(summaries)
summary_pivot = summary_df.set_index('Datacenter').T.reset_index()
summary_pivot.columns.values[0] = 'Metric'
display(Markdown(summary_pivot.to_markdown(index=False)))
#tools.display_dataframe_to_user(name="CPU DVFS Method Summary", dataframe=summary_pivot)

| Metric                          |     Datacenter_1 |     Datacenter_2 |     Datacenter_3 |
|:--------------------------------|-----------------:|-----------------:|-----------------:|
| Total Active Hosts (Running)    |    367           |    600           |    400           |
| Total Hosts Powered On          |   1200           |    600           |    400           |
| Total VMs Parsed                | 362544           | 514080           | 353184           |
| Total MIPS                      |      3.20544e+08 |      2.6712e+08  |      1.98432e+08 |
| Used MIPS                       |      1.38072e+08 |      1.98383e+08 |      1.41824e+08 |
| Total CPU Power (W)             |      6.79822e+06 |      6.40769e+06 |      4.69602e+06 |
| Total RAM Power (W)             | 195680           | 367700           | 425498           |
| Total BW Power (W)              | 647791           | 570428           | 423894           |
| Total I/O Power (W)             | 379390           | 325636           | 250756           |
| Total Disk I/O Throughput MB    |      3.75589e+06 |      3.68176e+06 |      3.62165e+06 |
| Total CPU Energy (kWh)          |    553.03        |    526.346       |    386.318       |
| Total RAM Energy (kWh)          |     15.9184      |     30.2039      |     35.0036      |
| Total BW Energy (kWh)           |     52.6973      |     46.8566      |     34.8716      |
| Total I/O Energy (kWh)          |     30.8807      |     26.7528      |     20.6391      |
| Total Power Consumption (kWh)   |    652.526       |    630.159       |    476.832       |
| Datacenter MIPS Usage (%)       |     43.0741      |     74.2672      |     71.4725      |
| Datacenter RAM Usage (%)        |     53.843       |     95.9007      |     96.4045      |
| Datacenter Network BW Usage (%) |      8.62158     |     15.3857      |     16.1755      |
| Datacenter Storage Usage (%)    |      5.26984     |      8.96703     |      8.29304     |